# Advanced RAG

In [1]:
# Import libraries
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from pathlib import Path

/home/nguyen/micromamba/envs/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from pathlib import Path
from llama_cpp import Llama

# 1. Point directly to your downloaded GGUF file inside the snapshot directory
# (Replace 'gemma-4-E2B-it-Q4_K_M.gguf' with the exact filename in your folder)
model_dir = Path("/home/nguyen/.cache/huggingface/hub/models--unsloth--gemma-4-E2B-it-GGUF/snapshots/90f9618340396838ee7ff5b0ba2da27da62953d3")
model_path = model_dir / "gemma-4-E2B-it-UD-Q4_K_XL.gguf"

# 2. Initialize the model (This handles both tokenizer and execution)
llm = Llama(
    model_path=str(model_path),
    n_ctx=4096,         # Context window limit to protect your 4GB VRAM
    n_gpu_layers=-1,    # -1 offloads ALL layers to your GPU
    verbose=False
)

# 3. Create your query rewriting prompt
system_prompt = "You are a RAG query optimizer. Rewrite the user input into clean keywords for a vector search. Output ONLY the rewritten query in JSON."
user_query = "Why loading gguf is faster than loading safetensor?"

# Combine using Gemma 4 chat structure format
prompt = f"""
<start_of_turn>user
{system_prompt}

Query: {user_query}<end_of_turn>
<start_of_turn>model
"""

# 4. Generate the optimized query
response = llm(prompt, max_tokens=64, stop=["<end_of_turn>"])
optimized_query = response["choices"][0]["text"].strip()

print("Rewritten Query:", optimized_query)

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


Rewritten Query: {"query": "Why loading gguf is faster than loading safetensor?"}
{"rewritten_query": "gguf loading speed vs safetensor loading speed"}


In [ ]:
# Install langchain for rewriting functions
!pip 

In [27]:
def rewrite_query(query):
    prompt = f"""
You are a retrieval query optimizer.

Given a user question, generate:
1. One improved search query.
2. Three alternative search queries

Return only JSON, repeat, only JSON

Question:
{query}

Answer: 
"""

    response = generate(prompt)

    return response

In [28]:
print(rewrite_query("What is Query Rewrite in RAG?"))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
    "improved_search_query": "What is Query Rewrite in RAG?",
    "alternative_search_queries": [
        "What is Query Rewrite in RAG?",
        "What is Query Rewrite in RAG?",
        "What is Query Rewrite in RAG?"
    ]
}
